# YOLOv8 Food Detection — FoodSafe
Trains YOLOv8n to detect Indian food items from camera feed.
Free model from Ultralytics. Dataset: Open Images v7 + custom Indian food images.

In [ ]:
# Install
# pip install ultralytics torch torchvision

from ultralytics import YOLO
import os

# Load pretrained YOLOv8 nano (smallest, fastest — good for mobile)
model = YOLO('yolov8n.pt')
print('Model loaded:', model.info())

In [ ]:
# Dataset config — Indian food classes
FOOD_CLASSES = [
    'turmeric_powder', 'chilli_powder', 'coriander_powder', 'cumin',
    'milk_packet', 'ghee_container', 'mustard_oil', 'coconut_oil',
    'rice', 'wheat_flour', 'dal_lentils', 'paneer',
    'honey_jar', 'spice_packet', 'loose_spice',
]

# Write dataset YAML
import yaml

dataset_config = {
    'path': '../data/food_detection',
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(FOOD_CLASSES),
    'names': FOOD_CLASSES,
}

with open('../data/food_detection.yaml', 'w') as f:
    yaml.dump(dataset_config, f)
print('Dataset config written')

In [ ]:
# Download Open Images food subset (free)
# Uses fiftyone — pip install fiftyone
import fiftyone as fo
import fiftyone.zoo as foz

food_labels = ['Turmeric', 'Milk', 'Rice', 'Honey', 'Cooking oil']

dataset = foz.load_zoo_dataset(
    'open-images-v7',
    split='train',
    label_types=['detections'],
    classes=food_labels,
    max_samples=500,
)
print(f'Downloaded {len(dataset)} samples')

In [ ]:
# Training — fine-tune on food dataset
# Free on Google Colab T4 GPU (~2 hours for 50 epochs)

results = model.train(
    data='../data/food_detection.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='foodsafe_yolo',
    project='../models',
    device='0',        # GPU; use 'cpu' if no GPU
    patience=10,       # Early stopping
    lr0=0.001,
    augment=True,
)

print('Training complete. Best model at:', results.save_dir)

In [ ]:
# Evaluate on test set
metrics = model.val(data='../data/food_detection.yaml', split='test')
print(f'mAP50:     {metrics.box.map50:.3f}')
print(f'mAP50-95:  {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.p.mean():.3f}')
print(f'Recall:    {metrics.box.r.mean():.3f}')

In [ ]:
# Export to ONNX for deployment in browser (TensorFlow.js compatible)
model.export(format='onnx', imgsz=640, simplify=True)
print('Exported to ONNX — ready for TensorFlow.js')

# Test inference on a single image
results = model.predict('../data/test_images/turmeric_sample.jpg', conf=0.5)
for r in results:
    for box in r.boxes:
        cls_id   = int(box.cls)
        cls_name = FOOD_CLASSES[cls_id]
        conf     = float(box.conf)
        print(f'Detected: {cls_name} ({conf:.2f})')